# 02. Model development and recalibration

Trains the EfficientNet-B0 classifier on the histopathology-confirmed dermoscopic development images, fits temperature scaling on the held-out development split, and writes predicted risks for the held-out and external images.

**Environment.** Kaggle notebook with a GPU (NVIDIA T4 in the reported run) and the same seven datasets attached as in notebook 01. Training takes about 40 minutes.

**Released model.** With `USE_RELEASED_MODEL = True` (default) the notebook loads `results/model/model_best.pt` and the temperature of the reported run and only scores images. Set it to `False` to train from ImageNet weights.

**Inputs.** `cohort_arm_a_endpoint.csv` and `cohort_arm_b_endpoint.csv`, read from `outputs/` if notebook 01 was run in this session, then from the output of notebook 01 attached as a Kaggle input, and otherwise from `results/cohorts/` in the repository.

**Outputs** (written to `outputs/`): `predictions_development_heldout.csv`, `predictions_external_armA.csv`, `predictions_external_armB.csv`, `model_best.pt` and `model_run_config.json`. The files from the reported run are in `results/predictions/` and `results/model/`, and its console output is in `results/logs/model_development_kaggle_log.txt`.

## 1. Setup

In [1]:
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo():
    candidates = [Path.cwd(), *Path.cwd().parents]
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):   # uploaded zips add a folder level
            candidates += sorted(kaggle.glob(depth))
    for p in candidates:
        if (p / "src" / "dermcal").is_dir():
            return p
    raise FileNotFoundError("Repository not found: attach it as a Kaggle dataset or run from inside it.")


REPO = find_repo()
sys.path.insert(0, str(REPO / "src"))
from dermcal.metrics import auc

SEED = 1234
IMG_SIZE = 224
BATCH = 64
EPOCHS = 4
LR = 3e-4
WEIGHT_DECAY = 1e-4
TRAIN_FRACTION = 0.85

DATA_ROOT = Path("/kaggle/input")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
CKPT = OUT / "model_best.pt"
RELEASED = REPO / "results" / "model"
# True: score with the weights and temperature of the reported run (no training).
# False: train from ImageNet weights and fit a new temperature.
USE_RELEASED_MODEL = True

random.seed(SEED); np.random.seed(SEED)

## 2. Cohorts and image files

In [2]:
def load_cohort(name):
    # 1. this session, 2. the output of notebook 01 attached as a Kaggle input,
    # 3. the cohort files of the reported run in the repository
    found = [OUT / name]
    if DATA_ROOT.exists():
        found += [p for p in sorted(DATA_ROOT.rglob(name)) if REPO not in p.parents]
    found.append(REPO / "results" / "cohorts" / name)
    for p in found:
        if p.exists():
            print(f"{name}: {p}")
            return pd.read_csv(p, low_memory=False)
    raise FileNotFoundError(f"{name} not found; run notebook 01 first")


arm_a = load_cohort("cohort_arm_a_endpoint.csv")     # histopathology-confirmed
arm_b = load_cohort("cohort_arm_b_endpoint.csv")     # all confirmation methods
print(f"Arm A: {len(arm_a):,}   Arm B: {len(arm_b):,}")

cohort_arm_a_endpoint.csv: /kaggle/input/notebooks/immuhammadsumairrazi/01-cohort/outputs/cohort_arm_a_endpoint.csv
cohort_arm_b_endpoint.csv: /kaggle/input/notebooks/immuhammadsumairrazi/01-cohort/outputs/cohort_arm_b_endpoint.csv
Arm A: 24,822   Arm B: 58,991


In [3]:
# Index every lesion image under the input folder by file stem. Folders holding
# segmentation masks or superpixel overlays are skipped.
EXCLUDE = ("groundtruth", "segmentation", "superpixel", "mask")
IMG_EXT = {".jpg", ".jpeg", ".png"}


def build_index(root):
    idx, dup = {}, 0
    for dirpath, _dirs, files in os.walk(root):
        if any(x in dirpath.lower() for x in EXCLUDE):
            continue
        for f in files:
            stem, ext = os.path.splitext(f)
            if ext.lower() not in IMG_EXT or stem.lower().endswith("_superpixels"):
                continue
            if stem in idx:
                dup += 1
                continue
            idx[stem] = os.path.join(dirpath, f)
    print(f"indexed {len(idx):,} images ({dup:,} duplicate file names ignored)")
    return idx


INDEX = build_index(DATA_ROOT)


def attach_paths(df, label):
    stems = df["image_id"].astype(str).map(lambda s: os.path.splitext(s)[0])
    df = df.copy()
    df["path"] = stems.map(INDEX.get)
    hit = df["path"].notna()
    print(f"  {label:<26} {hit.sum():>6,}/{len(df):>6,} images found ({100 * hit.mean():.1f}%)")
    return df[hit].reset_index(drop=True)


print("matching cohort rows to image files:")
dev = attach_paths(arm_a[arm_a["role"] == "development"], "development (Arm A)")
ext_a = attach_paths(arm_a[arm_a["role"] == "external_test"], "external (Arm A)")
ext_b = attach_paths(arm_b[arm_b["role"] == "external_test"], "external (Arm B)")

indexed 81,545 images (10,735 duplicate file names ignored)
matching cohort rows to image files:
  development (Arm A)        19,506/22,872 images found (85.3%)
  external (Arm A)            1,950/ 1,950 images found (100.0%)
  external (Arm B)           19,480/19,483 images found (100.0%)


In the reported run 19,506 of the 22,872 development images were found. The 3,366 missing images are ISIC 2019 test images that are not included in the Kaggle mirror (see notebook 03, section 8).

## 3. Grouped split

Images are grouped by lesion identifier where one exists (ISIC 2019), otherwise by patient identifier (ISIC 2020), otherwise by image. Groups are shuffled with a fixed seed and 85% are assigned to training; no group appears in both sets.

In [4]:
def group_key(r):
    for c in ("lesion_id", "patient_id"):
        v = r.get(c)
        if pd.notna(v) and str(v) not in ("", "nan"):
            return f"{r['source']}::{c}::{v}"
    return f"{r['source']}::image::{r['image_id']}"


dev["group"] = dev.apply(group_key, axis=1)
groups = dev["group"].unique()
rng = np.random.default_rng(SEED)
rng.shuffle(groups)
cut = int(TRAIN_FRACTION * len(groups))
train_g, val_g = set(groups[:cut]), set(groups[cut:])
train_df = dev[dev["group"].isin(train_g)].reset_index(drop=True)
val_df = dev[dev["group"].isin(val_g)].reset_index(drop=True)
assert not (set(train_df["group"]) & set(val_df["group"]))
print(f"train {len(train_df):,} images / {len(train_g):,} groups   "
      f"held-out {len(val_df):,} / {len(val_g):,}   "
      f"held-out proportion malignant {val_df['label_malignant'].mean():.3f}")

train 16,686 images / 7,878 groups   held-out 2,820 / 1,391   held-out proportion malignant 0.445


## 4. Model and training

EfficientNet-B0 with ImageNet weights and a single output unit. Images are resized to 224 × 224 pixels. Training uses AdamW with a one-cycle learning-rate schedule, mixed precision, and binary cross-entropy with the positive class weighted by the ratio of benign to malignant training images. The checkpoint with the highest held-out AUC is kept.

In [5]:
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees=20, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


class Lesions(Dataset):
    def __init__(self, df, tf):
        self.p = df["path"].tolist()
        self.y = df["label_malignant"].astype(float).tolist()
        self.tf = tf

    def __len__(self):
        return len(self.p)

    def __getitem__(self, i):
        img = Image.open(self.p[i]).convert("RGB")
        return self.tf(img), torch.tensor(self.y[i], dtype=torch.float32)


def loader(df, tf, shuffle):
    return DataLoader(Lesions(df, tf), batch_size=BATCH, shuffle=shuffle,
                      num_workers=2, pin_memory=(DEV == "cuda"), drop_last=False)


def make_model():
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    return m.to(DEV)


@torch.no_grad()
def predict(model, df):
    # returns logits
    model.eval()
    out = []
    for x, _ in loader(df, tf_eval, False):
        with torch.autocast("cuda", enabled=(DEV == "cuda")):
            out.append(model(x.to(DEV)).float().squeeze(1).cpu())
    return torch.cat(out).numpy()

device: cuda Tesla T4


In [6]:
model = make_model()
if USE_RELEASED_MODEL:
    model.load_state_dict(torch.load(RELEASED / "model_best.pt", map_location=DEV))
    print("loaded the weights of the reported run; training skipped")
elif CKPT.exists():
    model.load_state_dict(torch.load(CKPT, map_location=DEV))
    print(f"loaded {CKPT}; training skipped")
else:
    pos_w = torch.tensor([(train_df["label_malignant"] == 0).sum() /
                          max(1, (train_df["label_malignant"] == 1).sum())], device=DEV)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    tl = loader(train_df, tf_train, True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, epochs=EPOCHS, steps_per_epoch=len(tl))
    scaler = torch.amp.GradScaler("cuda", enabled=(DEV == "cuda"))
    best = -1
    for ep in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); run = 0.0
        for i, (x, y) in enumerate(tl, 1):
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=(DEV == "cuda")):
                loss = crit(model(x).squeeze(1), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
            if i % 50 == 0:
                print(f"  epoch {ep} step {i}/{len(tl)} loss {run / i:.4f}", flush=True)
        a = auc(val_df["label_malignant"], predict(model, val_df))
        print(f"epoch {ep}: loss {run / len(tl):.4f}  held-out AUC {a:.4f}  ({time.time() - t0:.0f}s)", flush=True)
        if a > best:
            best = a
            torch.save(model.state_dict(), CKPT)
            print("  saved checkpoint")
    model.load_state_dict(torch.load(CKPT, map_location=DEV))
    print(f"best held-out AUC {best:.4f}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 197MB/s]


loaded the weights of the reported run; training skipped


## 5. Temperature scaling

Weighting the positive class during training shifts the raw outputs, so a single temperature is fitted to the held-out logits by minimising the log loss. The model is not updated on any external data.

In [7]:
val_logit = predict(model, val_df)


def fit_temperature(logits, y):
    lg = torch.tensor(logits, dtype=torch.float32)
    yy = torch.tensor(np.asarray(y, dtype=float), dtype=torch.float32)
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    lossf = nn.BCEWithLogitsLoss()

    def step():
        opt.zero_grad()
        loss = lossf(lg / torch.exp(logT), yy)
        loss.backward()
        return loss

    opt.step(step)
    return float(torch.exp(logT).item())


TEMP = fit_temperature(val_logit, val_df["label_malignant"])
print(f"temperature fitted in this session {TEMP:.3f}")
if USE_RELEASED_MODEL:
    TEMP = json.loads((RELEASED / "model_run_config.json").read_text())["temperature"]
    print(f"temperature of the reported run   {TEMP:.3f} (used)")

temperature fitted in this session 1.343
temperature of the reported run   1.343 (used)


## 6. Predicted risks

In [8]:
def score(df):
    df = df.copy()
    df["risk"] = 1 / (1 + np.exp(-predict(model, df) / TEMP))
    return df


COLS = ["image_id", "source", "fst_group", "fitzpatrick", "label_malignant", "confirm_histo", "risk"]
for d, name in [(val_df, "development_heldout"), (ext_a, "external_armA"), (ext_b, "external_armB")]:
    score(d)[COLS].to_csv(OUT / f"predictions_{name}.csv", index=False)

meta = {"seed": SEED, "img_size": IMG_SIZE, "epochs": EPOCHS, "batch": BATCH,
        "lr": LR, "temperature": TEMP, "n_train": len(train_df), "n_val": len(val_df),
        "model": "efficientnet_b0 ImageNet1k", "min_events_for_slope": 50,
        "n_bootstrap": 2000, "weights": "released" if USE_RELEASED_MODEL else "trained in this session"}
(OUT / "model_run_config.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

{
  "seed": 1234,
  "img_size": 224,
  "epochs": 4,
  "batch": 64,
  "lr": 0.0003,
  "temperature": 1.3427139520645142,
  "n_train": 16686,
  "n_val": 2820,
  "model": "efficientnet_b0 ImageNet1k",
  "min_events_for_slope": 50,
  "n_bootstrap": 2000,
  "weights": "released"
}
